The main selling point of asyncio is the 'single-threaded' concurrency model based on our event loop..

This conventional approach only works in cases where single-threaded concurrency makes sense... There's 2 main scenarios where it doesn't.

### 1. Running CPU-bound code

We know why it doesn't make sense here... CPU-bound work involves running code... and concurrency when it comes to running code cannot be achieved on a single thread, thanks to the GIL.

We need to have some multi-processing here... 

Asyncio has support for multi-processing, so we can still get our work done via asyncio, but we'll have to use the multi-processing submodule along with it (it's integrated in asyncio), and tell asyncio to run such tasks in a process pool

In [ ]:
# running cpu bound work concurrently with plain tasks..

import asyncio
from util import delay, async_timed

@async_timed()
async def cpu_bound_work() -> int:
    counter = 0
    for i in range(100000000):
        counter = counter + 1
    return counter

@async_timed()
async def main():
    task_one = asyncio.create_task(cpu_bound_work())
    task_two = asyncio.create_task(cpu_bound_work())
    await task_one
    await task_two

asyncio.run(main())

# it doesn't work, they're still executing sequentially, because the fn is cpu-bound, only one of the 2 functions can every do its job at a time, leaving no possibility for concurrency..

# this can be mitigated by using the multiprocessing submodule of asyncio

looking above, you may think that adding tasks to cpu-bound work may not give concurrency, but it's no harm either, right?

Well, problem with cpu-bound code is that it hogs control, starving tasks that might actually benefit from single-threaded concurrency.

In [ ]:
import asyncio
from util import async_timed, delay

@async_timed()
async def cpu_bound_work() -> int:
    counter = 0
    for i in range(100000000):
        counter = counter + 1
    return counter

@async_timed()
async def main():
    task_one = asyncio.create_task(cpu_bound_work())
    task_two = asyncio.create_task(cpu_bound_work())
    delay_task = asyncio.create_task(delay(4)) # thanks to 2 cpu-bound tasks featuring ahead in queue of delay_task, delay_task must wait until they are done for processing, even though it can be run concurrently in a single-threaded model..
    await task_one
    await task_two
    await delay_task

asyncio.run(main())

### 2. Running blocking API's

You can't customize all api calling code to just work with coroutines / tasks... Chances are they're using blocking sockets instead of non-blocking sockets. If that's the case, then they too will hog control like cpu-bound code...

libraries like requests, time, DO use blocking sockets... use these libraries only for scripts... For production code catering to many api calls, which can be made concurrent, think of aiohttp.. (we'll use aiohttp properly further in this book)

As a general rule - 

1. any code that performs IO which isn't already a coroutine
2. any code performing cpu intensive work

is blocking.

In [ ]:
# trying to run concurrent calls using requests

import asyncio
import requests
from util import async_timed

@async_timed()
async def get_example_status() -> int:
    return requests.get('http:/ / www .example .com').status_code

@async_timed()
async def main():
    task_1 = asyncio.create_task(get_example_status())
    task_2 = asyncio.create_task(get_example_status())
    task_3 = asyncio.create_task(get_example_status())
    await task_1
    await task_2
    await task_3

asyncio.run(main()) # ~0.17s

# the wait times add up, inspite of using tasks.. this is because requests library uses blocking sockets..

In [ ]:
# trying to run concurrent calls using aiohttp

import asyncio
import aiohttp
from util import async_timed

@async_timed()
async def get_example_status(session: aiohttp.ClientSession) -> int:
    response = await session.get('http://www.example.com')
    return response.status

@async_timed()
async def main():
    async with aiohttp.ClientSession() as session:
        task_1 = asyncio.create_task(get_example_status(session))
        task_2 = asyncio.create_task(get_example_status(session))
        task_3 = asyncio.create_task(get_example_status(session))
        await task_1
        await task_2
        await task_3

asyncio.run(main()) # ~0.054s

# we'll learn more about aiohttp later.

note: we can still use requests, but then we'll have to run our tasks in a thread pool using a thread pool executor..